# 07 · Laser phase noise

A coherent receiver mixes the signal with a local oscillator (LO) laser.
Neither laser has a perfectly stable frequency, and the received carrier
phase wanders randomly.  This is the impairment that carrier-phase recovery
([11](11-blind-carrier-phase-recovery.ipynb), [12](12-pilot-aided-carrier-recovery.ipynb)) removes, and the
quantity that the characterization notebooks
([20](20-carrier-phase-characterization.ipynb)-[23](23-laser-linewidth-self-homodyne-iq.ipynb)) measure.

1. the **Wiener** (white frequency noise) model and its linewidth;
2. the **Lorentzian** field spectrum;
3. what phase noise does to symbols: the linewidth-symbol-period product;
4. single and dual polarization: a shared LO;
5. **flicker** (1/f) frequency noise and the FM-noise spectrum.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()


def host(a):
    return to_device(a, "cpu")

## 1. The Wiener model

The simplest laser model has a **white instantaneous frequency** noise.
The phase is its integral, a random walk:

$$\varphi[n] = \varphi[n-1] + \delta_n, \qquad
\delta_n \sim \mathcal{N}\!\left(0, \; 2\pi \Delta\nu / f_s\right),$$

so the phase variance grows linearly in time, `Var[φ(t) - φ(0)] = 2πΔν·t`.
`Δν` is the **linewidth**.  `generate_phase_noise` returns the trajectory
itself (in radians, float64), which keeps the ground truth at hand;
`apply_phase_noise` applies one to a signal.

In [ ]:
fs = 1e9
n = 20_000
fig, ax = plt.subplots(figsize=(8, 4))
for lw in (100e3, 1e6, 10e6):
    phi = host(
        ck.impairments.generate_phase_noise(
            num_samples=n, sampling_rate=fs, linewidth=lw, rng=1, device=DEVICE
        )
    )
    ax.plot(np.arange(n) / fs * 1e6, phi, label=f"Δν = {lw / 1e6:g} MHz")
ax.set_xlabel("Time [µs]")
ax.set_ylabel("Phase [rad]")
ax.legend()
plt.show()

# The variance law, over an ensemble of 500 independent lasers:
lw = 1e6
ens = host(
    ck.impairments.generate_phase_noise(
        num_samples=n,
        sampling_rate=fs,
        linewidth=lw,
        num_channels=500,
        rng=2,
        device=DEVICE,
    )
)
for k in (1_000, 10_000, 19_999):
    t = k / fs
    print(
        f"t = {t * 1e6:5.1f} µs: Var = {np.var(ens[:, k] - ens[:, 0]):.4f} rad², "
        f"2πΔν·t = {2 * np.pi * lw * t:.4f} rad²"
    )

## 2. The Lorentzian line

The optical field `E(t) = exp(jφ(t))` of a Wiener laser has a **Lorentzian**
power spectrum,

$$S_E(f) = \frac{1}{\pi}\,\frac{\Delta\nu / 2}{f^2 + (\Delta\nu/2)^2},$$

whose full width at half maximum (FWHM) is exactly `Δν`.  This is the
textbook meaning of "linewidth".

In [ ]:
fs, lw = 100e6, 1e6
phi = ck.impairments.generate_phase_noise(
    num_samples=2**21, sampling_rate=fs, linewidth=lw, rng=3, device=DEVICE
)
field = ck.backend.get_array_module(phi).exp(1j * phi).astype("complex64")
f, p = ck.spectral.welch_psd(field, sampling_rate=fs, nperseg=2**15)
f, p = host(f), host(p)
order = np.argsort(f)
f, p = f[order], p[order]
lorentz = (lw / 2) / np.pi / (f**2 + (lw / 2) ** 2)

above = f[p >= p.max() / 2]
print(
    f"measured FWHM {(above.max() - above.min()) / 1e6:.2f} MHz (Δν = {lw / 1e6:.0f} MHz, "
    f"bin {fs / 2**15 / 1e3:.1f} kHz)"
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(f / 1e6, 10 * np.log10(p), label="simulated field")
ax.plot(f / 1e6, 10 * np.log10(lorentz), "--", label="Lorentzian, FWHM = Δν")
ax.set_xlim(-10, 10)
ax.set_ylim(10 * np.log10(lorentz.max()) - 35, 10 * np.log10(lorentz.max()) + 3)
ax.set_xlabel("Frequency offset [MHz]")
ax.set_ylabel("PSD [dB/Hz]")
ax.legend()
plt.show()

## 3. What it does to symbols

What matters for a receiver is the phase change over the time it averages,
measured in symbol periods: the **linewidth-symbol-period product**
`Δν·T`.  Over `N` symbols the phase spreads by `sqrt(2π Δν T N)`
radians.  At 32 GBd a 100 kHz laser has `Δν·T = 3e-6`: invisible over a
few symbols, but over 10⁵ symbols the phase has walked a full radian, so
the constellation of an uncorrected record is a ring.  Carrier recovery
estimates the phase over windows short enough that it is nearly constant
inside them.

In [ ]:
C16 = Constellation.qam(16)
tx = ck.generate(C16, num_symbols=2**16, symbol_rate=32e9, rng=4, device=DEVICE)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, lw in zip(axes, (10e3, 100e3, 1e6)):
    r = ck.impairments.apply_phase_noise(tx, linewidth=lw, rng=5)
    print(
        f"Δν = {lw / 1e3:6.0f} kHz: Δν·T = {lw / tx.symbol_rate:.1e}, "
        f"phase spread over the record {np.sqrt(2 * np.pi * lw / tx.symbol_rate * 2**16):.2f} rad, "
        f"over 64 symbols {np.sqrt(2 * np.pi * lw / tx.symbol_rate * 64):.4f} rad"
    )
    ck.plotting.plot_constellation(
        r, ax=ax, title=f"Δν = {lw / 1e3:g} kHz, uncorrected"
    )
plt.show()

## 4. One LO, two polarizations

A dual-polarization receiver mixes both polarizations with the **same**
LO, so both see the same phase noise (`shared_lo=True`).  Carrier recovery
can then estimate one phase from both channels (`joint_channels=True` in
the CPR objects), with twice the symbols per estimate.  Independent
transmitters (or independent lasers per channel) give independent phases.

In [ ]:
dual = ck.generate(
    C16, num_symbols=2**14, symbol_rate=32e9, num_channels=2, rng=6, device=DEVICE
)
for shared in (True, False):
    r = ck.impairments.apply_phase_noise(dual, linewidth=1e6, shared_lo=shared, rng=7)
    # The phase each channel received, against its own transmitted symbols:
    ph = np.unwrap(np.angle(host(r.samples) * np.conj(host(dual.samples))), axis=-1)
    print(
        f"shared_lo={shared!s:5}: max phase difference between channels {np.max(np.abs(ph[0] - ph[1])):.3f} rad"
    )

## 5. Flicker noise and the FM-noise spectrum

Real lasers add **1/f (flicker) frequency noise** from technical sources:
current-source noise, temperature, acoustics.  Instead of a single
linewidth, the complete description is the one-sided frequency-noise PSD,

$$S_f(f) = \underbrace{\Delta\nu / \pi}_{\text{white FM}} + \underbrace{h_{-1} / f}_{\text{flicker}} \quad [\text{Hz}^2/\text{Hz}],$$

a flat plateau at `Δν/π` plus a rise toward low frequencies, crossing at
the corner `f_c = π h₋₁ / Δν`.  `generate_phase_noise(flicker=h₋₁)` draws
both, and `analysis.fm_noise_psd` estimates `S_f` from a phase trajectory.
With flicker, the measured width of the line grows with the observation
time, which is why the characterization notebooks report `S_f` rather than
one number.

In [ ]:
fs, lw, h = 100e6, 100e3, 1e9
phi = ck.impairments.generate_phase_noise(
    num_samples=2**20,
    sampling_rate=fs,
    linewidth=lw,
    flicker=h,
    flicker_f_min=1e3,
    rng=8,
    device=DEVICE,
)
f, s_f = ck.analysis.fm_noise_psd(phi, sampling_rate=fs, nperseg=2**16)
f, s_f = host(f), host(s_f)
fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(f[1:], s_f[1:], label="estimated from the phase")
ax.loglog(
    f[1:], lw / np.pi + h / f[1:], "--", label=r"model $\Delta\nu/\pi + h_{-1}/f$"
)
ax.axvline(
    np.pi * h / lw,
    color="gray",
    ls=":",
    label=f"corner f_c = {np.pi * h / lw / 1e3:.0f} kHz",
)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"$S_f$ [Hz²/Hz]")
ax.legend()
plt.show()

**Summary**

* White frequency noise makes the phase a random walk with
  `Var = 2πΔν·t`; the field line is Lorentzian with FWHM `Δν`.
* For a receiver the relevant number is `Δν·T` and the window it averages
  over.
* Dual-polarization receivers share the LO phase noise (`shared_lo=True`).
* Flicker noise adds a `1/f` rise to the FM-noise PSD; report `S_f(f)`.

**Next:** [08 · Front-end impairments](08-front-end-impairments.ipynb).